In [1]:
%pip install hf_xet

   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   - -------------------------------------- 0.1/2.8 MB 655.4 kB/s eta 0:00:05
   - -------------------------------------- 0.1/2.8 MB 726.2 kB/s eta 0:00:04
   -- ------------------------------------- 0.2/2.8 MB 748.1 kB/s eta 0:00:04
   --- ------------------------------------ 0.2/2.8 MB 885.4 kB/s eta 0:00:03
   --- ------------------------------------ 0.3/2.8 MB 850.6 kB/s eta 0:00:03
   ----- ---------------------------------- 0.4/2.8 MB 995.6 kB/s eta 0:00:03
   ------- -------------------------------- 0.5/2.8 MB 1.3 MB/s eta 0:00:02
   --------- ------------------------------ 0.7/2.8 MB 1.5 MB/s eta 0:00:02
   ---------- ----------------------------- 0.7/2.8 MB 1.5 MB/s eta 0:00:02
   ---------- ----------------------------- 0.7/2.8 MB 1.5 MB/s eta 0:00:02
   ---------- -------

Vector Database Setup (locally with Chroma)

Loads processed chunks, generates embeddings, and creates Chroma vector database.
- Uses sentence-transformers for embeddings
- Sets up Chroma with persistence
- Optimizes for retrieval performance
- Prepares for RAG pipeline

In [ ]:
import os
import json
import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer
from pathlib import Path
from typing import Dict, List
from tqdm import tqdm
import numpy as np

class VectorDatabaseBuilder: # build and manage Chroma vector db

    
    def __init__(self, 
                 persist_directory: str = "D:/PsyWiz/vector_db",
                 embedding_model: str = "sentence-transformers/all-MiniLM-L6-v2",
                 collection_name: str = "psywiz_papers"):
        
        self.persist_directory = Path(persist_directory)
        self.persist_directory.mkdir(parents=True, exist_ok=True)
        
        self.collection_name = collection_name
        
        print(f"Loading embedding model: {embedding_model}")
        self.embedding_model = SentenceTransformer(embedding_model)
        
        self.client = chromadb.PersistentClient(
            path=str(self.persist_directory),
            settings=Settings(
                anonymized_telemetry=False,
                allow_reset=True
            )
        )
        
        try:
            self.collection = self.client.get_collection(name=collection_name)
            print(f"Loaded existing collection: {collection_name}")
        except:
            self.collection = self.client.create_collection(
                name=collection_name,
                metadata={"description": "PsyWiz medical research papers chunks"}
            )
            print(f"Created new collection: {collection_name}")
    
    def load_chunks(self, chunks_file: str) -> List[Dict]:
        """Load processed chunks from JSON file"""
        print(f"Loading chunks from: {chunks_file}")
        
        with open(chunks_file, 'r', encoding='utf-8') as f:
            chunks = json.load(f)
        
        print(f"Loaded {len(chunks)} chunks")
        return chunks
    
    def generate_embeddings(self, texts: List[str], batch_size: int = 32) -> np.ndarray:

        print(f"Generating embeddings for {len(texts)} chunks...")
        
        # process in batches for memory efficiency
        all_embeddings = []
        
        for i in tqdm(range(0, len(texts), batch_size), desc="Embedding batches"):
            batch_texts = texts[i:i + batch_size]
            batch_embeddings = self.embedding_model.encode(
                batch_texts,
                convert_to_numpy=True,
                show_progress_bar=False
            )
            all_embeddings.append(batch_embeddings)
        
        embeddings = np.vstack(all_embeddings)
        print(f"Generated embeddings shape: {embeddings.shape}")
        
        return embeddings
    
    def prepare_chunk_data(self, chunks: List[Dict]) -> tuple:
        chunk_ids = []
        documents = []
        metadatas = []
        
        for chunk in chunks:
            chunk_ids.append(chunk['chunk_id'])
            documents.append(chunk['content'])
            
            metadata = {
                'document_id': chunk['metadata']['document_id'],
                'section': chunk['metadata']['section'],
                'section_index': chunk['metadata']['section_index'],
                'chunk_index': chunk['metadata']['chunk_index'],
                'source_file': chunk['metadata']['source_file'],
                'paper_title': chunk['metadata']['paper_title'],
                'doi': chunk['metadata']['doi'],
                'publication_date': chunk['metadata']['publication_date'],
                'source_url': chunk['metadata']['source_url'],
                'token_count': chunk['token_count']
            }
            
            if chunk['metadata']['authors']:
                metadata['authors'] = ', '.join(chunk['metadata']['authors'])
            else:
                metadata['authors'] = ''
            
            metadatas.append(metadata)
        
        return chunk_ids, documents, metadatas
    
    def add_chunks_to_collection(self, chunks: List[Dict], batch_size: int = 100): # add chunks to Chroma collection with embeddings

        
        existing_count = self.collection.count()
        if existing_count > 0:
            print(f"Collection already contains {existing_count} documents")
            user_input = input("Do you want to clear and rebuild? (y/N): ")
            if user_input.lower() == 'y':
                self.client.delete_collection(self.collection_name)
                self.collection = self.client.create_collection(
                    name=self.collection_name,
                    metadata={"description": "PsyWiz medical research papers chunks"}
                )
                print("Collection cleared and recreated")
            else:
                print("Skipping ingestion. Collection unchanged.")
                return
        
        chunk_ids, documents, metadatas = self.prepare_chunk_data(chunks)
        
        embeddings = self.generate_embeddings(documents)
        
        # add to collection in batches
        print(f"Adding {len(chunks)} chunks to collection...")
        
        for i in tqdm(range(0, len(chunks), batch_size), desc="Adding to Chroma"):
            end_idx = min(i + batch_size, len(chunks))
            
            batch_ids = chunk_ids[i:end_idx]
            batch_documents = documents[i:end_idx]
            batch_metadatas = metadatas[i:end_idx]
            batch_embeddings = embeddings[i:end_idx].tolist()
            
            self.collection.add(
                ids=batch_ids,
                documents=batch_documents,
                metadatas=batch_metadatas,
                embeddings=batch_embeddings
            )
        
        print(f"Successfully added {len(chunks)} chunks to collection")
        print(f"Total documents in collection: {self.collection.count()}")
    
    def test_retrieval(self, query: str, n_results: int = 5):
        """Test vector similarity search"""
        print(f"\nTesting retrieval with query: '{query}'")
        
        results = self.collection.query(
            query_texts=[query],
            n_results=n_results,
            include=['documents', 'metadatas', 'distances']
        )
        
        print(f"\nTop {n_results} results:")
        for i, (doc, metadata, distance) in enumerate(zip(
            results['documents'][0],
            results['metadatas'][0], 
            results['distances'][0]
        )):
            print(f"\n{i+1}. Distance: {distance:.4f}")
            print(f"   Section: {metadata['section']}")
            print(f"   Document: {metadata['document_id']}")
            print(f"   Content: {doc[:200]}...")
    
    def get_collection_stats(self):
        
        count = self.collection.count()
        
        if count > 0:
            sample = self.collection.get(limit=min(100, count), include=['metadatas'])
            
            doc_counts = {}
            section_counts = {}
            
            for metadata in sample['metadatas']:
                doc_id = metadata['document_id']
                section = metadata['section']
                
                doc_counts[doc_id] = doc_counts.get(doc_id, 0) + 1
                section_counts[section] = section_counts.get(section, 0) + 1
            
            print(f"\n{'='*50}")
            print("VECTOR DATABASE STATISTICS")
            print(f"{'='*50}")
            print(f"Total chunks: {count}")
            print(f"Documents: {len(doc_counts)}")
            print(f"Avg chunks per document: {np.mean(list(doc_counts.values())):.1f}")
            print(f"Most common sections:")
            for section, count in sorted(section_counts.items(), key=lambda x: x[1], reverse=True)[:5]:
                print(f"  - {section}: {count} chunks")

def main():
    """Main execution function"""
    
    vector_db = VectorDatabaseBuilder(
        persist_directory="D:/PsyWiz/vector_db",
        embedding_model="sentence-transformers/all-MiniLM-L6-v2",
        collection_name="psywiz_papers"
    )
    
    chunks_file = "D:/PsyWiz/chunks/all_chunks.json"
    chunks = vector_db.load_chunks(chunks_file)
    
    vector_db.add_chunks_to_collection(chunks, batch_size=50)
    
    vector_db.get_collection_stats()
    
    test_queries = [
        "depression prevalence in diabetes patients",
        "anxiety symptoms in adolescents",
        "food insecurity and mental health"
    ]
    
    for query in test_queries:
        vector_db.test_retrieval(query, n_results=3)
    
    print(f"\n{'='*50}")
    print("VECTOR DATABASE SETUP COMPLETE!")
    print(f"{'='*50}")
    print(f"Database location: {vector_db.persist_directory}")
    print(f"Collection name: {vector_db.collection_name}")
    print("Ready for RAG pipeline!")

if __name__ == "__main__":
    main()

Loading embedding model: sentence-transformers/all-MiniLM-L6-v2
Created new collection: psywiz_papers
Loading chunks from: D:/PsyWiz/chunks/all_chunks.json
Loaded 87 chunks
Generating embeddings for 87 chunks...


Embedding batches: 100%|██████████| 3/3 [00:05<00:00,  1.70s/it]


Generated embeddings shape: (87, 384)
Adding 87 chunks to collection...


Adding to Chroma: 100%|██████████| 2/2 [00:00<00:00,  4.45it/s]


Successfully added 87 chunks to collection
Total documents in collection: 87

VECTOR DATABASE STATISTICS
Total chunks: 87
Documents: 2
Avg chunks per document: 43.5
Most common sections:
  - sectional study conducted in Lahore, Pakistan. BioScientific Review 4(4),: 12 chunks
  - Cambridge Prisms: Global Mental Health: 9 chunks
  - The results of the quality assessment, including: 5 chunks
  - Bodyweight categories: 4 chunks
  - Food diversity and insecurity: 3 chunks

Testing retrieval with query: 'depression prevalence in diabetes patients'

Top 3 results:

1. Distance: 0.6746
   Section: The results of the quality assessment, including
   Document: article_2
   Content: *Three estimates are included for Lloyd et al 2018 for data from Bangla-
desh, India, and Pakistan. Common mental disorders in Bangladesh, India and Pakistan
www.jogh.org • doi: 10.7189/jogh.09.020417...

2. Distance: 0.7076
   Section: Some studies provided multiple estimates, either for multiple NCDs or for anxiety 

---
# DB BROWSER

Check what files exist

In [ ]:

import os
from pathlib import Path

db_path = Path("D:/PsyWiz/vector_db")
print("Database files:")
for file in db_path.rglob("*"):
    if file.is_file():
        size_mb = file.stat().st_size / (1024*1024)
        print(f"  {file.name}: {size_mb:.1f} MB")

Database files:
  chroma.sqlite3: 1.6 MB
  data_level0.bin: 16.0 MB
  header.bin: 0.0 MB
  length.bin: 0.0 MB
  link_lists.bin: 0.0 MB


Test persistence - connect to existing DB

In [ ]:
test_db = VectorDatabaseBuilder()
print(f"Existing chunks: {test_db.collection.count()}")
test_db.test_retrieval("depression", n_results=2)

Loading embedding model: sentence-transformers/all-MiniLM-L6-v2
Loaded existing collection: psywiz_papers
Existing chunks: 87

Testing retrieval with query: 'depression'

Top 2 results:

1. Distance: 0.9411
   Section: journal of
   Document: article_2
   Content: health
global
© 2019 The Author(s)
JoGH © 2019 ISoGH
Common mental disorders and non-communicable
diseases
The WHO estimated that 4.4% of the global population was living
with depression and 3.6% was ...

2. Distance: 1.0477
   Section: The mean age of participants was 12.9 ± 1.6 years in boys and 12.0 ±
   Document: article_3
   Content: Specifically, dietary diver-
sity, food insecurity and folate deficiency were significantly associ-
ated with greater depression symptoms in both sexes (respectively,
for boys, IRR = 0.906; CI 95% = 0...
